<a href="https://colab.research.google.com/github/9z2fqjckjv-bot/ColabTool/blob/main/%E3%83%87%E3%83%BC%E3%82%BF%E5%85%A5%E5%8A%9B%EF%BC%86%E3%82%BF%E3%82%A4%E3%83%94%E3%83%B3%E3%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Googleドライブの同期設定とライブラリのインストール
モデルキャッシュとタイピング履歴を保存するために、Googleドライブをマウントし、Gradioおよび必要なパッケージをインストールします。

In [1]:
!pip install -q gradio transformers accelerate

In [2]:
!pip install -q reportlab

In [3]:
from google.colab import drive
import os

# Googleドライブをマウント
drive.mount('/content/drive')

# モデルのキャッシュ先と履歴の保存先ディレクトリをGoogleドライブ内に作成
DRIVE_DIR = '/content/drive/MyDrive/typing_tool'
os.makedirs(DRIVE_DIR, exist_ok=True)

CACHE_DIR = os.path.join(DRIVE_DIR, 'llm_cache')
HISTORY_PATH = os.path.join(DRIVE_DIR, 'typing_history.csv')

print(f"モデル保存先: {CACHE_DIR}")
print(f"履歴保存先: {HISTORY_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
モデル保存先: /content/drive/MyDrive/typing_tool/llm_cache
履歴保存先: /content/drive/MyDrive/typing_tool/typing_history.csv


### 2. Qwen2.5 LLMのロード
Googleドライブ上のキャッシュディレクトリを利用して、Qwen2.5 (Qwen/Qwen2.5-0.5B-Instruct など軽量モデル) を読み込みます。初回のみダウンロードに時間がかかりますが、2回目以降はGoogleドライブから即時にロードされます。

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import shutil

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("LLMモデルをロードしています... (初回はダウンロードが実行されます)")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=CACHE_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        cache_dir=CACHE_DIR
    )
except Exception as e:
    print(f"ロード中にエラーが発生しました: {e}")
    print("Googleドライブ上の壊れたキャッシュファイルを削除し、再ダウンロードを試みます...")

    # キャッシュディレクトリが破損している場合は削除して再作成
    if os.path.exists(CACHE_DIR):
        shutil.rmtree(CACHE_DIR)
    os.makedirs(CACHE_DIR, exist_ok=True)

    # 再度ロード実行
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=CACHE_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        cache_dir=CACHE_DIR
    )

print("モデルのロードが完了しました。")

LLMモデルをロードしています... (初回はダウンロードが実行されます)
ロード中にエラーが発生しました: Error while deserializing header: header too large
Googleドライブ上の壊れたキャッシュファイルを削除し、再ダウンロードを試みます...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

モデルのロードが完了しました。


In [5]:
def generate_typing_prompt(theme="日常的な話題や豆知識、IT、歴史、科学など多種多様なジャンル"):
    """Qwen2.5を使用して、必ず『1文だけ』の短い日本語のタイピング練習用お題を生成します"""
    prompt = f"タイピング練習用として、次のテーマに沿った、短くて自然な日本語の文章を『必ず1文だけ（句点「。」で終わる文章）』作成してください。改行、解説、記号、導入用の言葉（例:『以下はお題です』など）は一切含めず、純粋な本文の1文のみを出力してください。\nテーマ: {theme}"
    messages = [
        {"role": "system", "content": "You are a strict task execution assistant. Output only the requested single sentence in Japanese. No explanations, no extra lines, no quotes."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.85
    )
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    # 不要な文字や複数行が出力された場合のクレンジング
    cleaned = response.strip().split('\n')[0].replace('"', '').replace('「', '').replace('』', '').replace('』', '').strip()
    return cleaned if cleaned else "タイピング技術を磨くために毎日練習を続けましょう。"

### 3. Gradio Webインターフェースの起動
外部ブラウザの新規タブ等で実行可能なGradioアプリケーションを起動します。タイピングスコアや結果、履歴はGoogleドライブへ自動保存されます。

In [6]:
import gradio as gr
import time
import os
import pandas as pd

# セッション状態管理用の辞書
state_data = {
    "current_target": "",
    "start_time": 0.0
}

def initialize_app():
    """アプリ起動時に最初のお題を自動生成してセットアップする"""
    first_target = generate_typing_prompt()
    state_data["current_target"] = first_target
    state_data["start_time"] = time.time()

    # 初期表示用の履歴読み込み
    if os.path.exists(HISTORY_PATH):
        try:
            df_history = pd.read_csv(HISTORY_PATH)
        except Exception:
            df_history = pd.DataFrame(columns=["日時", "お題", "解答", "速度(文字/分)", "合計文字数", "精密度(%)", "総合点数"])
    else:
        df_history = pd.DataFrame(columns=["日時", "お題", "解答", "速度(文字/分)", "合計文字数", "精密度(%)", "総合点数"])

    return first_target, "", "タイピングを開始してください！", df_history.tail(10)

def process_and_next(user_input):
    """ユーザーが入力した内容を判定し、スコアを記録した後、自動的に次のお題を生成する"""
    end_time = time.time()
    target = state_data["current_target"]
    start_time = state_data["start_time"]

    if not target or start_time == 0:
        # お題が空だった場合のフォールバック
        target = "練習用テキストです。"
        start_time = time.time() - 5

    elapsed_time = end_time - start_time
    if elapsed_time <= 0:
        elapsed_time = 0.1

    # 1. 各種パラメーターの計算
    # 合計文字数（お題の長さ）
    total_chars = len(target)

    # 正しい文字数（位置一致で判定）
    correct_chars = sum(1 for c1, c2 in zip(user_input, target) if c1 == c2)

    # 精密度（正しい文字数 ÷ 合計文字数 の百分率）
    accuracy = (correct_chars / total_chars) * 100 if total_chars > 0 else 0

    # 入力速度（1分あたりの文字数：ユーザー入力文字数 / 経過時間 * 60）
    cpm = (len(user_input) / elapsed_time) * 60

    # 総合点数（速度と精密度の掛け合わせによる簡易スコア算出）
    total_score = int(cpm * (accuracy / 100))

    # ログ作成
    new_log = {
        "日時": time.strftime("%Y-%m-%d %H:%M:%S"),
        "お題": target,
        "解答": user_input,
        "速度(文字/分)": round(cpm, 1),
        "合計文字数": total_chars,
        "精密度(%)": round(accuracy, 1),
        "総合点数": total_score
    }

    # Googleドライブの履歴CSVに追記保存
    if os.path.exists(HISTORY_PATH):
        try:
            df_history = pd.read_csv(HISTORY_PATH)
        except Exception:
            df_history = pd.DataFrame(columns=["日時", "お題", "解答", "速度(文字/分)", "合計文字数", "精密度(%)", "総合点数"])
    else:
        df_history = pd.DataFrame(columns=["日時", "お題", "解答", "速度(文字/分)", "合計文字数", "精密度(%)", "総合点数"])

    df_history = pd.concat([df_history, pd.DataFrame([new_log])], ignore_index=True)
    df_history.to_csv(HISTORY_PATH, index=False, encoding='utf-8-sig')

    # 判定結果テキスト表示用
    result_text = (
        f"【前回の結果】\n"
        f"・お題: {target}\n"
        f"・解答: {user_input}\n"
        f"・入力速度: {cpm:.1f} 文字/分 (CPM)\n"
        f"・合計文字数: {total_chars} 文字\n"
        f"・精密度: {accuracy:.1f} %\n"
        f"・総合点数: {total_score} 点"
    )

    # 自動的に『新しいお題（次の1文）』を生成して、次のセッションを開始する
    next_target = generate_typing_prompt()
    state_data["current_target"] = next_target
    state_data["start_time"] = time.time()

    return next_target, "", result_text, df_history.tail(10)

# Gradio UIデザイン構築
with gr.Blocks(title="AI無限タイピング練習") as demo:
    gr.Markdown("# ⌨️ Qwen2.5搭載 無限タイピング & データ入力練習")
    gr.Markdown("送信（判定）を行うと、自動でスコアが計算され、間髪入れずに次の新しい1文のお題が自動生成されます。")

    target_display = gr.Textbox(label="現在のお題（タイピングしてください）", interactive=False, text_align="left")
    user_textbox = gr.Textbox(label="ここに入力してEnterまたは送信ボタンを押してください", placeholder="タイピング入力...", interactive=True)

    submit_btn = gr.Button("🏁 送信 / 判定して次のお題へ", variant="success")

    result_display = gr.Textbox(label="最新の判定結果", interactive=False)

    gr.Markdown("### 📊 タイピング練習累積履歴（Googleドライブに保管されています）")
    history_table = gr.Dataframe(interactive=False)

    # アプリ起動時に最初のお題を自動セットアップ
    demo.load(fn=initialize_app, outputs=[target_display, user_textbox, result_display, history_table])

    # 決定イベントの設定（送信ボタン、またはテキストボックスでのEnterキー入力両対応）
    submit_btn.click(
        fn=process_and_next,
        inputs=[user_textbox],
        outputs=[target_display, user_textbox, result_display, history_table]
    )
    user_textbox.submit(
        fn=process_and_next,
        inputs=[user_textbox],
        outputs=[target_display, user_textbox, result_display, history_table]
    )

# 新しいタブで実行可能な共有リンクを生成
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f07851322534158208.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_58452/717971640.py:80: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_history = pd.concat([df_history, pd.DataFrame([new_log])], ignore_index=True)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f07851322534158208.gradio.live
